# 02 — Adaptive Regularization using Cubics (ARC)

Ratio-test adaptation of $M_k$. Sensitivity to $M_0$ and rejected-step counts.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic, Rosenbrock, LogSumExp
from cubic_reg.solvers import cr, arc

%matplotlib inline

In [ ]:
problems = [
    Quadratic(n=40, condition=100.0, seed=0),
    LogSumExp(n=15, m=40, mu=0.1, seed=0),
    Rosenbrock(n=2),
]
rows = []
for p in problems:
    x0 = np.ones(p.dim) if p.dim > 2 else np.array([-1.2, 1.0])
    r_cr = cr.minimize(p, x0=x0, M=1.0, eps=1e-6, max_iter=200)
    r_arc = arc.minimize(p, x0=x0, M0=1.0, eps=1e-6, max_iter=200)
    rows.append((p.name, r_cr.nit, r_cr.grad_norm, r_arc.nit, r_arc.grad_norm, r_arc.rejected_steps))
print(f"{'problem':35} {'CR nit':>8} {'CR ||g||':>12} {'ARC nit':>8} {'ARC ||g||':>12} {'rej':>5}")
for row in rows:
    print(f"{row[0]:35} {row[1]:8d} {row[2]:12.3e} {row[3]:8d} {row[4]:12.3e} {row[5]:5d}")

In [ ]:
p = Quadratic(n=40, condition=100.0, seed=1)
x0 = np.ones(p.dim)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for M0 in [1e-4, 1e-2, 1.0, 1e2, 1e4]:
    r = arc.minimize(p, x0=x0, M0=M0, eps=1e-8, max_iter=150)
    gap = np.maximum(np.asarray(r.history_f) - p.f_star, 1e-16)
    axes[0].semilogy(gap, label=f"M0={M0:g}")
    axes[1].semilogy(r.history_M, label=f"M0={M0:g}")
axes[0].set_title("gap vs iter"); axes[1].set_title("M_k vs iter")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
p = Quadratic(n=60, condition=50.0, seed=0)
r = arc.minimize(p, x0=np.ones(p.dim), M0=1.0, mode="krylov", krylov_dim=15, eps=1e-6)
print("ARC+Krylov", r.success, r.nit, r.grad_norm, "rejected", r.rejected_steps, "hvp", r.n_hvp)